# Ray checkpointing example

This notebook runs a **Ray Train** checkpointing demo on **Red Hat OpenShift AI** using the CodeFlare SDK:

- Submit a **RayJob** with a **managed Ray cluster** (`ManagedClusterConfig`) so KubeRay lifecycles the cluster with the job (`shutdownAfterJobFinishes`).
- Configure **S3 access** via an OpenShift AI **Data Connection** attached to this workbench (see below).
- **Monitor** training in the Ray dashboard (**Jobs** tab), then **suspend and resume** the RayJob (`job.stop()` / `job.resubmit()`) to verify training **resumes from S3** after a simulated interruption.

For quota/preemption with **Red Hat build of Kueue**, see the companion blog and OpenShift *AI workloads* documentation.

Training script: `train_with_checkpoints.py` in this directory (same source as the CodeFlare SDK guided demo).

## Import required libraries

In [ ]:
import os
import subprocess
import time

from codeflare_sdk import RayJob, ManagedClusterConfig, set_api_client, get_cluster
from codeflare_sdk.common.utils.constants import CUDA_PY312_RUNTIME_IMAGE
from codeflare_sdk.common.utils.utils import get_ray_image_for_python_version
from kube_authkit import AuthConfig, get_k8s_client


def resolve_ray_image() -> str:
    """Pick a MODH Ray image; fall back to py312 when workbench Python has no default."""
    if os.environ.get("RAY_IMAGE"):
        return os.environ["RAY_IMAGE"]
    image = get_ray_image_for_python_version()
    return image or CUDA_PY312_RUNTIME_IMAGE

## Authenticate to your OpenShift cluster

In [ ]:
# Authenticate to your Kubernetes/OpenShift cluster using kube-authkit

# Option 1: Auto-detect credentials (kubeconfig or in-cluster service account)
# NOTE: In RHOAI Workbenches the workbench service account may not have Ray RBAC
# permissions. Use Option 2 (token) unless your admin has granted SA permissions
# (see RHOAIENG-46748). Auto-detect works if you have a local kubeconfig.
# auth_config = AuthConfig(method="auto")

# Option 2 (Recommended for RHOAI Workbenches): Token-based authentication
# Get your token with: oc whoami -t  (or from the OpenShift console → Copy login command)
auth_config = AuthConfig(
    method="openshift",
    k8s_api_host="https://api.example.com:6443",
    token="sha256~XXXXX",  # oc whoami -t
)

# Option 3: OIDC authentication (for BYOIDC-enabled clusters)
# auth_config = AuthConfig(
#     method="oidc",
#     k8s_api_host="https://api.example.com:6443",
#     oidc_issuer="https://your-oidc-provider.com",
#     client_id="your-client-id",
#     use_device_flow=True,  # Interactive device flow for notebook environments
# )

api_client = get_k8s_client(config=auth_config)
# Set to False for self-signed / dev API certificates (optional).
api_client.configuration.verify_ssl = False
set_api_client(api_client)

NAMESPACE = "your-namespace"  # Data Science Project where the RayJob runs
JOB_NAME = "checkpointing-job"


def get_odh_training_jobs_url(namespace: str) -> str:
    """OpenShift AI Training jobs page for Pause/Resume/Delete in the UI."""
    try:
        base = subprocess.check_output(
            ["oc", "get", "consolelink", "rhodslink", "-o", "jsonpath={.spec.href}"],
            text=True,
        ).strip().rstrip("/")
        return f"{base}/develop-train/training-jobs/{namespace}"
    except Exception:
        return f"<your-openshift-ai-dashboard>/develop-train/training-jobs/{namespace}"


print(f"Namespace: {NAMESPACE!r}, RayJob name: {JOB_NAME!r}")

## Configure S3 credentials

Create an **S3 Data Connection** in your OpenShift AI project and attach it to this workbench before running the notebook. Credential keys are injected as environment variables and copied into the RayJob `runtime_env` below.

For temporary credentials (SSO/federated), set `AWS_SESSION_TOKEN` in the notebook environment before running the next cell — the Data Connection UI does not collect session tokens.

In [ ]:
AWS_ENV_KEYS = (
    "AWS_ACCESS_KEY_ID",
    "AWS_SECRET_ACCESS_KEY",
    "AWS_DEFAULT_REGION",
    "AWS_S3_BUCKET",
    "AWS_SESSION_TOKEN",  # optional; required for temporary credentials
)
AWS_CREDENTIALS = {k: os.environ[k] for k in AWS_ENV_KEYS if os.environ.get(k)}

required = ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_S3_BUCKET")
missing = [k for k in required if k not in AWS_CREDENTIALS]
if missing:
    raise ValueError(
        f"Missing AWS env vars: {missing}. Attach an S3 Data Connection to this workbench "
        "(Settings → Environment variables) or export the variables manually."
    )

if "AWS_DEFAULT_REGION" not in AWS_CREDENTIALS:
    AWS_CREDENTIALS["AWS_DEFAULT_REGION"] = "us-east-1"

# Change this if you see a Ray version mismatch when resuming from S3.
CHECKPOINT_PREFIX = os.environ.get("RAY_CHECKPOINT_PREFIX", "mnist-demo")

print(f"Using bucket: {AWS_CREDENTIALS['AWS_S3_BUCKET']}")
print(f"S3 checkpoint prefix: ray-checkpoints/{CHECKPOINT_PREFIX}")

## Submit RayJob (managed Ray cluster)

In [ ]:
ray_image = resolve_ray_image()
print(f"Using Ray image: {ray_image}")

managed = ManagedClusterConfig(
    num_workers=2,
    head_cpu_requests=2,
    head_cpu_limits=4,
    head_memory_requests=4,
    head_memory_limits=8,
    worker_cpu_requests=2,
    worker_cpu_limits=4,
    worker_memory_requests=4,
    worker_memory_limits=8,
    image=ray_image,
)

job = RayJob(
    job_name=JOB_NAME,
    entrypoint="python train_with_checkpoints.py",
    cluster_config=managed,
    namespace=NAMESPACE,
    runtime_env={
        "working_dir": "./",
        "pip": ["torch", "torchvision", "s3fs", "pyarrow"],
        "env_vars": {
            **AWS_CREDENTIALS,
            "RAY_CHECKPOINT_PREFIX": CHECKPOINT_PREFIX,
            "RAY_TRAIN_WORKER_GROUP_START_TIMEOUT_S": "120",  # Allow time for worker scheduling
        },
    },
)

job.submit()
print(job.status())
print(f"RayCluster template name: {job.cluster_name}")
print("Watch logs for: NO CHECKPOINT FOUND - Starting fresh")

## Monitor job progress (status + Ray dashboard Jobs)

Poll `job.status()`, then open the **Ray dashboard** URL from the RayCluster created by your RayJob. Use **Jobs** in the dashboard for live driver logs (epochs, checkpoint messages).

In [ ]:
print(job.status())
# KubeRay assigns a generated name — not job.cluster_name (the template).
cluster = None
ray_cluster_name = None
for _ in range(36):
    status_data = job._api.get_job_status(name=job.name, k8s_namespace=NAMESPACE)
    ray_cluster_name = (status_data or {}).get("rayClusterName")
    if ray_cluster_name:
        cluster = get_cluster(ray_cluster_name, namespace=NAMESPACE, verify_tls=False)
        if cluster is not None:
            break
    time.sleep(5)
if cluster is None:
    raise RuntimeError(
        "RayCluster not ready — check RayJob status and KubeRay operator logs."
    )
print(f"Ray Dashboard (open in browser): {cluster.cluster_dashboard_uri()}")
print(f"OpenShift AI — Training jobs: {get_odh_training_jobs_url(NAMESPACE)}")
print(f"Find RayJob '{JOB_NAME}' in the list to use Pause / Resume / Delete.")
print("In the Ray dashboard, open Jobs and stream logs for the training driver.")
print(
    "Wait for at least one full epoch and a checkpoint to S3 before running the suspend cell."
)

## Suspend RayJob (checkpoint demo)

After logs show at least **one epoch** and a checkpoint written to **S3**, suspend the RayJob.

Use **Pause** in the OpenShift AI UI (Training jobs URL printed above), or run the next cell (`job.stop()`).

In [ ]:
print("=" * 60)
print("SUSPENDING RayJob (checkpoint demo — not deleting the RayJob CR)")
print("Checkpoints remain in S3.")
print(f"OpenShift AI — Training jobs: {get_odh_training_jobs_url(NAMESPACE)}")
print("=" * 60)

job.stop()
print("Stop requested; poll job.status() until the RayJob reports suspended / non-running.")

## Resume RayJob

Use **Resume** in the OpenShift AI UI, or run `job.resubmit()` in the next cell. When the RayCluster is back, confirm in the dashboard **Jobs** view: `RESUMING FROM CHECKPOINT - Starting at epoch N`.

In [ ]:
print("=" * 60)
print("RESUMING RayJob after suspend")
print("Watch for RESUMING FROM CHECKPOINT in dashboard Jobs logs")
print("=" * 60)

job.resubmit()
time.sleep(10)
print(job.status())

## Verify resume from checkpoint

In the Ray dashboard **Jobs** tab, look for:

```
RESUMING FROM CHECKPOINT - Starting at epoch N
Previous loss: X.XXXX
```

That confirms optimizer and progress were restored from S3 across the suspend/resume cycle.

In [ ]:
print(job.status())
try:
    status_data = job._api.get_job_status(name=job.name, k8s_namespace=NAMESPACE)
    ray_cluster_name = (status_data or {}).get("rayClusterName")
    if ray_cluster_name:
        cluster = get_cluster(ray_cluster_name, namespace=NAMESPACE, verify_tls=False)
        if cluster is not None:
            print(f"Ray Dashboard: {cluster.cluster_dashboard_uri()}")
        else:
            print(f"RayCluster {ray_cluster_name!r} not found yet.")
    else:
        print("RayCluster name not assigned yet — check job.status().")
except Exception as e:
    print(f"Could not resolve cluster yet: {e}")
print("Check Jobs tab for: RESUMING FROM CHECKPOINT - Starting at epoch N")

## Cleanup

Use **Delete** in the OpenShift AI UI or execute the next cell to delete the RayJob and tear down the RayCluster if it is still present.

In [ ]:
print("Cleaning up...")
try:
    status_data = job._api.get_job_status(name=job.name, k8s_namespace=NAMESPACE)
    ray_cluster_name = (status_data or {}).get("rayClusterName")
except Exception:
    ray_cluster_name = None

try:
    job.delete()
except Exception:
    pass

if ray_cluster_name:
    try:
        c = get_cluster(ray_cluster_name, namespace=NAMESPACE, verify_tls=False)
        if c is not None:
            c.down()
    except Exception:
        pass

print("Cleanup attempted (RayJob delete; cluster.down if RayCluster still exists).")

In [ ]:
# No explicit logout needed - authentication is managed automatically by kube-authkit